# Diagnose output concentration across qubit widths

Use a factory of compiled QNodes to study a fixed circuit family from 4 to 16 qubits. Compare the sample variance of global parity with its population variance. The diagnostic resamples parameters rather than fitting them to training data.

Install QASP 0.1.3 as shown in the [tutorial setup guide](../README.md), then run the cells in order.

In [ ]:
%matplotlib inline
from importlib.metadata import version
import numpy as np
import torch
import matplotlib.pyplot as plt
from IPython.display import display

print("QASP", version("qasp"))
np.random.seed(17)
torch.manual_seed(17)

## Define the circuit family

Keep depth and connectivity fixed as rules across widths. Each circuit uses a scaled Y encoding, one layer of trainable Y rotations and a chain of CZ gates. The measured observable is the product of Z on all wires. At each width the factory passes a compiled circuit to the diagnoser. For drawing, it is convenient to return the compiled builder itself; both forms work in QASP 0.1.2.

In [ ]:
from qasp import AngleAnsatz, AngleEmbedding, CircuitBuilder, Ones, Uniform, barren_plateau_diagnoser

def build_circuit(n):
    embedding = AngleEmbedding(n, features=n, rotations="Y", input_scaling=True)
    ansatz = AngleAnsatz(n, feature_map=embedding, rotations="Y",
                         include_imprimitive=True, imprimitive="CZ", imprimitive_pattern="chain")
    builder = CircuitBuilder(n)
    builder.set_ansatz(ansatz)
    parity = "Z" * n + " " + " ".join(map(str, range(n)))
    builder.set_measurement("expectation", observables=[parity])
    builder.build()
    builder.compile(device="default.qubit")
    return builder

## Draw an ensemble at every width

Use a fixed zero input of the appropriate length, vary the angles uniformly on [−π, π], and keep encoding scales at one. Here n_samples counts independent draws **per width**. The complete sweep evaluates 896 initializations. output_index=0 selects the one measured moment.

In [ ]:
widths = [4, 6, 8, 10, 12, 14, 16]
settings = {
    "initialization": Uniform(-np.pi, np.pi),
    "input_scaling_init": Ones(),
    "n_samples": 128, "seed": 19,
}
sweep = barren_plateau_diagnoser(
    model_factory=build_circuit, qubit_list=widths,
    state_factory=lambda n: torch.zeros(n, dtype=torch.float64),
    output_index=0, **settings,
)

## Compare measured and analytic variance

The zero input makes the encoding rotations identities. CZ commutes with global Z, so the measured cost is $C_n=\prod_{j=1}^n\cos\theta_j$. Independent uniform angles have $E[\cos\theta_j]=0$ and $E[\cos^2\theta_j]=1/2$, hence $\mathrm{Var}(C_n)=2^{-n}$. The measured markers can fluctuate around this line with 128 draws. This statistic describes output concentration; a gradient-variance study requires gradient samples as well.

In [ ]:
width = sweep["qubit_list"]
observed = sweep["cost_variance"]
exact = 2.0 ** (-width)
np.testing.assert_array_equal(width, widths)
np.testing.assert_allclose(observed, sweep["expectation_value_variance"])
print("qubits   measured variance   population variance")
for n, v, ref in zip(width, observed, exact):
    print(f"{n:6d}   {v:17.6g}   {ref:19.6g}")
plt.semilogy(width, observed, "o-", label="128 draws per width")
plt.semilogy(width, exact, "--", label="Population variance")
plt.xticks(width)
plt.xlabel("Number of qubits")
plt.ylabel("Variance of global Z parity")
plt.legend()
plt.show()

## Inspect one member of the family

Draw the four-wire circuit with a representative set of angles. This shows the architecture being sampled. Each point on the variance curve aggregates many draws.

In [ ]:
import pennylane as qml

example = build_circuit(4)
x = torch.zeros(4, dtype=torch.float64)
args = {"params": torch.linspace(0.1, 0.4, 4, dtype=torch.float64),
        "input_scaling_params": torch.ones_like(x)}
fig, ax = qml.draw_mpl(example.qnode)(x, **args)
display(fig)
plt.close(fig)

### Preview the representative circuit

`to_tikz` returns the LaTeX source. Pass it to QASP’s `show_tikz(tex)` to preview the circuit inline. The preview uses temporary files. Use `save_tikz` on the builder when you want to keep a source file.

In [ ]:
from qasp import show_tikz

tex = example.to_tikz(x, **args, standalone=True, symbolic_parameters=True)
show_tikz(tex)

## Try it

Double the sample count and repeat with a new seed. Then change the observable to Z on the first wire. Derive the new population variance before reusing the reference curve.